In [1]:
import sys
import os

sys.path.append(
    os.path.abspath(".")
)

In [2]:
sys.path.append(
    os.path.abspath("..")
)

In [ ]:
from src.datasets.brats_dataset import create_file_list
from src.config import DATA_DIR
from src.transforms.preprocessing import (
    train_transform,
    val_transform
)

In [4]:
files = create_file_list(DATA_DIR)

print(len(files))

369


In [5]:
from sklearn.model_selection import train_test_split

files = create_file_list(DATA_DIR)

# 70% train
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# 15% val, 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_files))
print("Val  :", len(val_files))
print("Test :", len(test_files))

Train: 258
Val  : 55
Test : 56


In [6]:
from monai.data import Dataset

train_ds = Dataset(
    data=train_files,
    transform=train_transform
)

val_ds = Dataset(
    data=val_files,
    transform=val_transform
)

test_ds = Dataset(
    data=test_files,
    transform=val_transform
)

In [7]:
from src.graphs.create_graph_dataset import create_graph_dataset

In [8]:
train_graphs = create_graph_dataset(train_ds)

val_graphs = create_graph_dataset(val_ds)

test_graphs = create_graph_dataset(test_ds)

  0%|          | 0/258 [00:00<?, ?it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  0%|          | 1/258 [00:09<39:52,  9.31s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 2/258 [00:13<26:10,  6.13s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 3/258 [

In [9]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=1,
    shuffle=False
)

In [ ]:
from train.train_gcn import train_gcn

model = train_gcn(
    train_loader,
    val_loader,
    epochs=200,
    lr=5e-4
)


Training Class Distribution:
Counter({np.int64(0): 89866, np.int64(2): 8551, np.int64(3): 1587, np.int64(1): 1181})

Class Weights:
tensor([ 0.2815, 21.4193,  2.9583, 15.9397])
Epoch 000 | Train Loss: 0.8962 | Train Acc: 0.2004 | Val Loss: 0.3567 | Val Acc: 0.6746
Epoch 010 | Train Loss: 0.6262 | Train Acc: 0.5494 | Val Loss: 0.2910 | Val Acc: 0.8275
Epoch 020 | Train Loss: 0.5883 | Train Acc: 0.5646 | Val Loss: 0.2681 | Val Acc: 0.8038
Epoch 030 | Train Loss: 0.5602 | Train Acc: 0.5880 | Val Loss: 0.2589 | Val Acc: 0.8166
Epoch 040 | Train Loss: 0.5278 | Train Acc: 0.5917 | Val Loss: 0.2654 | Val Acc: 0.8328
Epoch 050 | Train Loss: 0.5207 | Train Acc: 0.5882 | Val Loss: 0.2756 | Val Acc: 0.8576

Early stopping at epoch 51

Training Complete
Best Validation Accuracy: 0.8759


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.eval()

all_true = []
all_pred = []

with torch.no_grad():

    for graph in val_loader:

        graph = graph.to(device)

        out = model(
            graph.x,
            graph.edge_index
        )

        pred = out.argmax(dim=1)

        all_true.extend(
            graph.y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")
print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3]
    )
)

              precision    recall  f1-score   support

           0       1.00      0.89      0.94    124497
           1       0.17      0.52      0.26       700
           2       0.22      0.71      0.33      4646
           3       0.16      0.65      0.25       693

    accuracy                           0.88    130536
   macro avg       0.39      0.69      0.45    130536
weighted avg       0.96      0.88      0.91    130536


Confusion Matrix:

[[110248   1136  11544   1569]
 [     3    366    134    197]
 [   149    589   3277    631]
 [    17     75    150    451]]


In [33]:
import numpy as np


def dice_score(y_true, y_pred, cls):

    y_true = (y_true == cls)
    y_pred = (y_pred == cls)

    intersection = np.sum(
        y_true & y_pred
    )

    return (
        2 * intersection
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + 1e-8
    )


print("\n===== Dice Scores =====")

for cls in range(4):

    dice = dice_score(
        np.array(all_true),
        np.array(all_pred),
        cls
    )

    print(
        f"Class {cls}: {dice:.4f}"
    )


===== Dice Scores =====
Class 0: 0.9386
Class 1: 0.2554
Class 2: 0.3318
Class 3: 0.2547


Better results and extended version on gcn after doing changes

In [ ]:
from train.train_gcn import train_gcn

model = train_gcn(
    train_loader,
    val_loader,
    epochs=200,
    lr=5e-4
)

Epoch 000 | Train Loss: 0.5346 | Train Acc: 0.9253 | Val Loss: 0.3756 | Val Acc: 0.9776
Epoch 010 | Train Loss: 0.4152 | Train Acc: 0.9527 | Val Loss: 0.3217 | Val Acc: 0.9762
Epoch 020 | Train Loss: 0.4008 | Train Acc: 0.9556 | Val Loss: 0.3180 | Val Acc: 0.9771
Epoch 030 | Train Loss: 0.3923 | Train Acc: 0.9574 | Val Loss: 0.3234 | Val Acc: 0.9758
Epoch 040 | Train Loss: 0.3829 | Train Acc: 0.9597 | Val Loss: 0.3206 | Val Acc: 0.9744
Epoch 050 | Train Loss: 0.3763 | Train Acc: 0.9603 | Val Loss: 0.3146 | Val Acc: 0.9805
Epoch 060 | Train Loss: 0.3701 | Train Acc: 0.9609 | Val Loss: 0.3179 | Val Acc: 0.9768

Early stopping at epoch 64

Training Complete
Best Validation Accuracy: 0.9839


In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.eval()

all_true = []
all_pred = []

with torch.no_grad():

    for graph in val_loader:

        graph = graph.to(device)

        out = model(
            graph.x,
            graph.edge_index
        )

        pred = out.argmax(dim=1)

        all_true.extend(
            graph.y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")
print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3]
    )
)

              precision    recall  f1-score   support

           0       0.99      1.00      0.99   1152995
           1       0.61      0.47      0.53      4124
           2       0.65      0.53      0.59     21233
           3       0.63      0.64      0.64      4094

    accuracy                           0.98   1182446
   macro avg       0.72      0.66      0.69   1182446
weighted avg       0.98      0.98      0.98   1182446


Confusion Matrix:

[[1147540      95    4795     565]
 [    899    1918     847     460]
 [   8504     914   11292     523]
 [    865     241     360    2628]]


In [12]:
import numpy as np


def dice_score(y_true, y_pred, cls):

    y_true = (y_true == cls)
    y_pred = (y_pred == cls)

    intersection = np.sum(
        y_true & y_pred
    )

    return (
        2 * intersection
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + 1e-8
    )


print("\n===== Dice Scores =====")

for cls in range(4):

    dice = dice_score(
        np.array(all_true),
        np.array(all_pred),
        cls
    )

    print(
        f"Class {cls}: {dice:.4f}"
    )


===== Dice Scores =====
Class 0: 0.9932
Class 1: 0.5261
Class 2: 0.5862
Class 3: 0.6356


In [17]:
from torch_geometric.loader import DataLoader
from evaluate_brats_regions import evaluate_brats_regions


test_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)


scores = evaluate_brats_regions(
    model,
    test_loader
)


===== BraTS Region Dice Scores =====
WT (Whole Tumor)      : 0.7093
TC (Tumor Core)       : 0.6743
ET (Enhancing Tumor)  : 0.6356


Splitting with train , test and validate and then doing evaluate

In [ ]:
from train.train_gcn import train_gcn

model = train_gcn(
    train_loader,
    val_loader,
    epochs=200,
    lr=5e-4
)

Epoch 000 | Train Loss: 0.5501 | Train Acc: 0.9221 | Val Loss: 0.3871 | Val Acc: 0.9685
Epoch 010 | Train Loss: 0.4097 | Train Acc: 0.9533 | Val Loss: 0.3303 | Val Acc: 0.9786
Epoch 020 | Train Loss: 0.3919 | Train Acc: 0.9558 | Val Loss: 0.3257 | Val Acc: 0.9775

Early stopping at epoch 26

Training Complete
Best Validation Accuracy: 0.9818


In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.eval()

all_true = []
all_pred = []

with torch.no_grad():

    for graph in test_loader:

        graph = graph.to(device)

        out = model(
            graph.x,
            graph.edge_index
        )

        pred = out.argmax(dim=1)

        all_true.extend(
            graph.y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")
print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3]
    )
)

              precision    recall  f1-score   support

           0       0.99      0.99      0.99    872959
           1       0.56      0.47      0.51      3721
           2       0.55      0.55      0.55     15202
           3       0.61      0.62      0.62      2942

    accuracy                           0.98    894824
   macro avg       0.68      0.66      0.67    894824
weighted avg       0.98      0.98      0.98    894824


Confusion Matrix:

[[866631    202   5494    632]
 [   745   1753   1025    198]
 [  5465    985   8433    319]
 [   569    177    375   1821]]


In [12]:
import numpy as np


def dice_score(y_true, y_pred, cls):

    y_true = (y_true == cls)
    y_pred = (y_pred == cls)

    intersection = np.sum(
        y_true & y_pred
    )

    return (
        2 * intersection
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + 1e-8
    )


print("\n===== Dice Scores =====")

for cls in range(4):

    dice = dice_score(
        np.array(all_true),
        np.array(all_pred),
        cls
    )

    print(
        f"Class {cls}: {dice:.4f}"
    )


===== Dice Scores =====
Class 0: 0.9925
Class 1: 0.5127
Class 2: 0.5525
Class 3: 0.6160


In [13]:
from torch_geometric.loader import DataLoader
from evaluate_brats_regions import evaluate_brats_regions


scores = evaluate_brats_regions(
    model,
    test_loader
)


===== BraTS Region Dice Scores =====
WT (Whole Tumor)      : 0.6972
TC (Tumor Core)       : 0.6195
ET (Enhancing Tumor)  : 0.6160
